In [ ]:
import torch, random, io, sys, warnings
import os, numpy as np, pandas as pd, pyreadr
from tqdm import tqdm

import sys, os
sys.path.append(os.path.abspath(".."))

from cpd_model import parse_args, learn_one_seq_penalty

warnings.filterwarnings("ignore")

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

# ======================================================
# CONFIG
# ======================================================
REPS = 10
TRUE_CP = [100, 200]
K_LIST = [10] 
TOL = 10

# ======================================================
# BASE ARGS
# ======================================================
base_args = parse_args()
base_args.epoch = 150
base_args.K_dim = 2  
base_args.z_dim = 3  
base_args.decoder_lr = 0.01
base_args.decoder_iteration = 20
base_args.langevin_s = 0.2
base_args.langevin_K = 100
base_args.kappa = 0.8
base_args.penalties = [0.01, 0.05, 0.1, 1]
base_args.nu_iteration = 20
base_args.output_layer = [50, 50]
base_args.scale_delta = False
base_args.signif_level = 0.99
base_args.true_CP_full = TRUE_CP

# ======================================================
# MAIN LOOP
# ======================================================
records = []

GLOBAL_SEED = 1

for K_dim in K_LIST:
    print(f"\n==============================")
    print(f" Running K_dim = {K_dim}")
    print(f"==============================")

    for rep in range(1, REPS + 1):

        print(f"\n--- Replicate {rep}/{REPS} ---")

        # ---------- seed ----------
        SEED = GLOBAL_SEED + rep
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

        # ---------- load data ----------
        Y = pyreadr.read_r(f"../real_data_sim/sim_dat_ult_5_{rep}.RDS")
        X = pyreadr.read_r(f"../real_data_sim/sim_x_ult_5_{rep}.RDS")

        Y_df = np.array(list(Y.values())[0])
        X_df = np.array(list(X.values())[0])

        # expand X
        X_rep = np.repeat(X_df[:, np.newaxis, :], 100, axis=1)
        Y = Y_df[:, :, 0:3]
        X = X_rep

        # ---------- args ----------
        args = parse_args()
        args.__dict__.update(base_args.__dict__)

        args.K_dim = K_dim 
        args.x_dim = X.shape[2]
        args.y_dim = Y.shape[2]
        args.num_time = X.shape[0]
        args.num_samples = X.shape[1]

        # ---------- tensor ----------
        x_input = torch.tensor(X, dtype=torch.float32).to(device)
        y_input = torch.tensor(Y, dtype=torch.float32).to(device)

        # ---------- split ----------
        odd_idx = range(1, args.num_time, 2)
        even_idx = range(0, args.num_time, 2)

        x_train = x_input[odd_idx].reshape(-1, args.x_dim)
        x_test  = x_input[even_idx].reshape(-1, args.x_dim)
        y_train = y_input[odd_idx].reshape(-1, args.y_dim)
        y_test  = y_input[even_idx].reshape(-1, args.y_dim)

        # ======================================================
        # penalty selection
        # ======================================================
        results_half = []

        for penalty in args.penalties:
            _stdout = sys.stdout
            # sys.stdout = io.StringIO()
            try:
                loss, pen = learn_one_seq_penalty(
                    args, x_train, y_train, x_test, y_test,
                    penalty=penalty, half=True
                )
            finally:
                sys.stdout = _stdout

            results_half.append([loss, pen])

        results_half = np.array(results_half)
        best_idx = np.argmin(results_half[:, 0])
        best_penalty = args.penalties[best_idx]

        print(f"[INFO] Best penalty = {best_penalty}")

        # ======================================================
        # full training
        # ======================================================
        _stdout = sys.stdout
        sys.stdout = io.StringIO()
        try:
            out = learn_one_seq_penalty(
                args,
                x_input.reshape(-1, args.x_dim),
                y_input.reshape(-1, args.y_dim),
                x_input.reshape(-1, args.x_dim),
                y_input.reshape(-1, args.y_dim),
                penalty=best_penalty,
                half=False
            )
            result = out[0]
        finally:
            sys.stdout = _stdout

        torch.cuda.empty_cache()

        # ======================================================
        # evaluation
        # ======================================================
        est_cp = np.array(result[5], dtype=int) if len(result[5]) > 0 else np.array([])
        true_cp = np.array(TRUE_CP)

        if len(est_cp) == 0:
            cover_rate = 0
            avg_dist = np.nan
            FP = 0
            FN = len(true_cp)
        else:
            dist_mat = np.abs(est_cp[:, None] - true_cp[None, :])
            min_dist_true = dist_mat.min(axis=0)
            min_dist_est  = dist_mat.min(axis=1)

            cover_rate = np.mean(min_dist_true <= TOL)
            avg_dist   = np.mean(min_dist_true)
            FP = np.sum(min_dist_est > TOL)
            FN = np.sum(min_dist_true > TOL)

        # ======================================================
        # record
        # ======================================================
        records.append({
            "K_dim": K_dim,
            "rep": rep,
            "best_penalty": best_penalty,
            "num_detected": len(est_cp),

            # core output
            "est_CP": str(list(est_cp)),
            "true_CP": str(TRUE_CP),

            # optional debug
            "CE": result[0],
        })

# ======================================================
# SAVE
# ======================================================
df = pd.DataFrame(records)
df.to_csv("cpd_Kdim_experiment_10.csv", index=False)


[INFO] Using device: cuda

 Running K_dim = 10

--- Replicate 1/10 ---
Epoch   5 | Loss=571.567627 | Kurtosis=4.866825
Epoch  10 | Loss=558.062866 | Kurtosis=9.745268
Epoch  15 | Loss=555.585938 | Kurtosis=7.836161
Epoch  20 | Loss=553.288452 | Kurtosis=5.204926
Epoch  25 | Loss=551.441467 | Kurtosis=6.588610
Epoch  30 | Loss=547.557251 | Kurtosis=6.667030
Epoch  35 | Loss=545.482544 | Kurtosis=9.931942
Epoch  40 | Loss=544.182068 | Kurtosis=16.424501
Epoch  45 | Loss=542.594666 | Kurtosis=9.346970
Epoch  50 | Loss=541.268127 | Kurtosis=13.083049
Epoch  55 | Loss=540.282593 | Kurtosis=13.154599
Epoch  60 | Loss=537.821167 | Kurtosis=10.979305
Epoch  65 | Loss=535.466309 | Kurtosis=10.692907
Epoch  70 | Loss=537.291992 | Kurtosis=10.595197
Epoch  75 | Loss=536.349548 | Kurtosis=13.197355
Epoch  80 | Loss=536.090210 | Kurtosis=11.887727
Epoch  85 | Loss=535.206299 | Kurtosis=11.623713
Epoch  90 | Loss=534.990234 | Kurtosis=12.409396
Epoch  95 | Loss=534.019470 | Kurtosis=10.867958
Epoch 

In [8]:
import numpy as np
import pandas as pd
import re

# =====================================================
# CONFIG
# =====================================================
CSV_PATH = "cpd_Kdim_experiment_2_3_7.csv"
TRUE_CP = [100, 200]
TOL = 10
T = 296


# =====================================================
# parse est_CP
# =====================================================
import ast

def parse_cp(x):
    if pd.isna(x):
        return []
    return list(ast.literal_eval(str(x)))


# =====================================================
# evaluation
# =====================================================
def evaluate_cp(est_cp, true_cp, tol, T):
    est_cp = np.array(est_cp, dtype=int)
    true_cp = np.array(true_cp, dtype=int)

    if len(est_cp) == 0:
        FP = 0
        FN = len(true_cp)
        Dt_e = np.inf
        De_t = np.inf
        CE = abs(len(est_cp) - len(true_cp))
        CS = 0.0
        return FP, FN, Dt_e, De_t, CE, CS

    dist_mat = np.abs(est_cp[:, None] - true_cp[None, :])

    min_dist_true = dist_mat.min(axis=0)
    min_dist_est  = dist_mat.min(axis=1)

    FP = np.sum(min_dist_est > tol)
    FN = np.sum(min_dist_true > tol)

    Dt_e = np.max(min_dist_true)
    De_t = np.max(min_dist_est)

    CE = abs(len(est_cp) - len(true_cp))

    # segmentation score
    def get_segments(cp, T):
        cp = np.sort(cp)
        segs = []
        prev = 0
        for c in cp:
            segs.append(set(range(prev, c)))
            prev = c
        segs.append(set(range(prev, T)))
        return segs

    G  = get_segments(true_cp, T)
    Gp = get_segments(est_cp, T)

    def jaccard(A, B):
        return len(A & B) / len(A | B) if len(A | B) > 0 else 0

    CS = 0
    for A in G:
        best = max(jaccard(A, Ap) for Ap in Gp)
        CS += len(A) * best
    CS /= T

    return FP, FN, Dt_e, De_t, CE, CS


# =====================================================
# LOAD + COMPUTE
# =====================================================
df = pd.read_csv(CSV_PATH)
df["est_CP"] = df["est_CP"].apply(parse_cp)

df = df.drop(columns=["CE"], errors="ignore")
# df["est_CP"] = df["est_CP"].apply(parse_cp)

metrics = df["est_CP"].apply(
    lambda cp: evaluate_cp(cp, TRUE_CP, TOL, T)
)

metrics_df = pd.DataFrame(
    metrics.tolist(),
    columns=["FP", "FN", "Dt_e", "De_t", "CE", "CS"]
)

df = pd.concat([df, metrics_df], axis=1)

summary = (
    df.groupby("K_dim")[["FP","FN","Dt_e","De_t","CE","CS"]]
    .mean()
    .round(2)
    .reset_index()
)

print(summary)


def to_latex_table(summary):
    rows = []

    best_FP = summary["FP"].min()
    best_FN = summary["FN"].min()
    best_Dte = summary["Dt_e"].min()
    best_Det = summary["De_t"].min()
    best_CE = summary["CE"].min()
    best_CS = summary["CS"].max()

    for _, r in summary.iterrows():
        def bold(val, best, is_max=False):
            if (val == best):
                return f"\\textbf{{{val:.2f}}}"
            return f"{val:.2f}"

        row = [
            int(r["K_dim"]),
            bold(r["FP"], best_FP),
            bold(r["FN"], best_FN),
            bold(r["Dt_e"], best_Dte),
            bold(r["De_t"], best_Det),
            bold(r["CE"], best_CE),
            bold(r["CS"], best_CS),
        ]
        rows.append(row)

    latex = []
    latex.append("\\begin{table}[!ht]")
    latex.append("\\caption{Sensitivity analysis with respect to $K$.}")
    latex.append("\\label{tab:kdim}")
    latex.append("\\centering")
    latex.append("\\small")
    latex.append("\\begin{tabular}{cccccccc}")
    latex.append("\\toprule")
    latex.append("$K$ & FP$\\downarrow$ & FN$\\downarrow$ & $D_{t\\to e}$ & $D_{e\\to t}$ & CE$\\downarrow$ & CS$\\uparrow$ \\\\")
    latex.append("\\midrule")

    for r in rows:
        latex.append(f"{r[0]} & {r[1]} & {r[2]} & {r[3]} & {r[4]} & {r[5]} & {r[6]} \\\\")

    latex.append("\\bottomrule")
    latex.append("\\end{tabular}")
    latex.append("\\end{table}")

    return "\n".join(latex)


latex_table = to_latex_table(summary)
print("\n===== LaTeX Table =====\n")
print(latex_table)

   K_dim   FP   FN  Dt_e  De_t   CE    CS
0      2  1.0  0.0   0.2  61.9  1.0  0.90
1      3  0.3  0.8  69.5  15.6  0.7  0.74
2      7  1.0  1.0  66.6  63.9  0.0  0.68

===== LaTeX Table =====

\begin{table}[!ht]
\caption{Sensitivity analysis with respect to $K$.}
\label{tab:kdim}
\centering
\small
\begin{tabular}{cccccccc}
\toprule
$K$ & FP$\downarrow$ & FN$\downarrow$ & $D_{t\to e}$ & $D_{e\to t}$ & CE$\downarrow$ & CS$\uparrow$ \\
\midrule
2 & 1.00 & \textbf{0.00} & \textbf{0.20} & 61.90 & 1.00 & \textbf{0.90} \\
3 & \textbf{0.30} & 0.80 & 69.50 & \textbf{15.60} & 0.70 & 0.74 \\
7 & 1.00 & 1.00 & 66.60 & 63.90 & \textbf{0.00} & 0.68 \\
\bottomrule
\end{tabular}
\end{table}
